In [20]:
"""
This script trains PU models for each training set and saves the models and predictions.
"""

import os

workdir = '/vast/ai_projects/Papenfuss_SVEnsemble2'
os.chdir(workdir)
os.getcwd()

import pickle # noqa: E402 (flake8)
import copy # noqa: E402 (flake8)
import importlib # noqa: E402 (flake8)
from pathlib import Path # noqa: E402 (flake8)
from collections import defaultdict # noqa: E402 (flake8)

import numpy as np # noqa: E402 (flake8)
import pandas as pd # noqa: E402 (flake8)
import matplotlib as mpl # noqa: E402 (flake8)
import pyranges as pr # noqa: E402 (flake8)
import sklearn # noqa: E402 (flake8)

from sklearn.ensemble import RandomForestClassifier as RF # noqa: E402 (flake8)
import sklearn.model_selection as skm # noqa: E402 (flake8)
from sklearn.metrics import confusion_matrix # noqa: E402 (flake8)

from svensemble import _caller_params as caller_params # noqa: E402 (flake8)
import svensemble.utils as sv_utils # noqa: E402 (flake8)
import svensemble.matched_breakpoints as mbps # noqa: E402 (flake8)
import svensemble.models as models # noqa: E402 (flake8)

importlib.reload(caller_params)
importlib.reload(sv_utils)
importlib.reload(mbps)
importlib.reload(models)

pu_savedir = Path('saved/pu')
pu_savedir.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)

joint_bp_df_path = Path('saved/matched/joint_bp_df.csv')

training_sets =  {
    'gridss': ['gridss'],
    'manta_candidate': ['manta_candidate'],
    'manta_diploid': ['manta_diploid'],
    'gridss+manta_candidate': ['gridss', 'manta_candidate'],
    'gridss+manta_diploid': ['gridss', 'manta_diploid']
}

training_set_presets = {
    'default': training_sets,
    'haploid': {
        'gridss': ['gridss'],
        'manta_candidate': ['manta_candidate'],
        'gridss+manta_candidate': ['gridss', 'manta_candidate']
    }
}

samples = {}
for sample_name in ['CHM13', 'NA12878', 'NA12878_30x']:
    samples.update(sv_utils.create_config_dict(sample_name))


In [ ]:
for sample_name, sample_config in samples.items():
    joint_bp_df = pd.read_csv(
        sample_config['sample_dir'] / 'sv_calling' / 'svensemble2' / 'matched_bps.csv',
        low_memory=False,
        index_col=0
    )

    training_set_dict = mbps.preprocess_sample_for_pu(
        joint_bp_df,
        sample_config,
        'gnomad'
    )
    del joint_bp_df

    training_set_train_test_split_dict = {}
    for training_set_name, training_df in training_set_dict.items():
        X, y = mbps.get_X_y_from_training_df(training_df, match_col='ref_match')
        X_train, X_val, y_train, y_val = skm.train_test_split(X, y, train_size=0.8, random_state=0)
        training_set_train_test_split_dict[training_set_name] = {'X': X, 'X_train': X_train, 'X_val': X_val, 'y': y, 'y_train': y_train, 'y_val': y_val}
    del training_set_dict

    unbiased_dict = {}
    for training_set_name, training_set_train_test_split in training_set_train_test_split_dict.items():
        RF_duplessis_fit, y_duplessis = models.train_unbiased_rf_for_split(training_set_train_test_split)
        # Set the index of y_duplessis to the index of the training set so it is possible to merge it with the set if needed
        y_duplessis.index = training_set_train_test_split['X'].index
        unbiased_dict[training_set_name] = {'RF_fit': RF_duplessis_fit, 'y': y_duplessis}
    del training_set_train_test_split_dict

    # Save the unbiased_dict
    with open(sample_config['sample_dir'] / 'sv_calling' / 'svensemble2' / 'unbiased_dict.pkl', 'wb') as f:
        pickle.dump(unbiased_dict, f)


In [39]:
# training_set_dict = {callset_name: None for callset_name in training_sets.keys()}

# for training_set_name, training_set_list in training_sets.items():
#     print(f'Processing {training_set_name} training set')
#     print(f'Training set contains {len(training_set_list)} callsets: {training_set_list}')
#     training_set = joint_bp_df
#     training_set.loc[:, 'ref_match'] = training_set['gnomad_INFO_EVENT'].notna()
#     training_set_col_starts_to_retain = []
#     for caller in training_set_list:
#         training_set_col_starts_to_retain.extend(cols_to_retain_dict[caller])
#     print(f'Training set column prefixes to retain: {training_set_col_starts_to_retain}')
#     training_set_cols_to_retain = []
#     for col_start_to_retain in training_set_col_starts_to_retain:
#         training_set_cols_to_retain.extend(training_set.columns[training_set.columns.str.startswith(col_start_to_retain)])
#     training_set_cols_to_retain.extend(cols_to_retain_universal)
#     training_set_cols_to_retain = list(set(training_set_cols_to_retain))
#     print(f'Training set columns to retain: {training_set_cols_to_retain}')
#     event_fields = [info_event_fields[caller] for caller in training_set_list]
#     training_set_dict[training_set_name] = training_set.loc[:, training_set_cols_to_retain][training_set[event_fields].notna().any(axis=1)]
#     print(f'Training set shape: {training_set_dict[training_set_name].shape}')

# # training_set_dict


In [ ]:
# Handle boolean columns that were incorrectly assigned as object type
for training_set_name, training_set in training_set_dict.copy().items():
    for column in training_set.columns:
        # Check if column contains boolean-like strings
        if training_set[column].dtype == 'object':
            try:
                # Check if column contains True/False strings
                if training_set[column].astype(str).str.contains(r'(True|False)', na=False).any():
                    # Convert to boolean, filling NAs with False
                    training_set_dict[training_set_name].loc[:, column] = (
                        training_set[column]
                        .map({'True': True, 'False': False})
                        .fillna('False')
                        .astype(bool)
                    )
                    # Force the dtype to be bool
                    training_set_dict[training_set_name] = training_set_dict[training_set_name].astype({column: bool})
            except (AttributeError, TypeError) as err:
                print(f'Column {column} does not support string operations: {err}')
                # Skip if column doesn't support string operations
                continue

# Verify conversion worked correctly
print("Column dtype:", training_set_dict['gridss']['gridss_INFO_IMPRECISE'].dtype)
training_set_dict['gridss']['gridss_INFO_IMPRECISE']

In [25]:
def pu_preprocess(overlapped_df, match_col='ref_match'):
    overlapped_df_copy = overlapped_df.copy().rename(columns={match_col: 'ref_match'})
    X = pd.get_dummies(overlapped_df_copy.drop(columns=['ref_match']))
    X = X.fillna(0)
    # Drop columns with only 1 unique value
    nunique = X.nunique()
    constant_cols = nunique[nunique == 1].index
    print(f'Dropping {len(constant_cols)} constant columns: {constant_cols}')
    X = X.drop(columns=constant_cols)
    y = overlapped_df_copy['ref_match']

    return X, y

In [ ]:
pu_dict = defaultdict(list)
for training_set_name, training_set in training_set_dict.items():
    X, y = pu_preprocess(training_set, match_col='ref_match')
    pu_dict[training_set_name] = (X, y)

In [ ]:
pu_dict['gridss'][0]

In [ ]:
pos_joint_bp_df = joint_bp_df[joint_bp_df.columns[joint_bp_df.columns.str.contains('POS_bnd')]]
pos_joint_bp_df = pos_joint_bp_df.dropna(axis=0)
pos_joint_bp_df

In [ ]:
# Do train/test split for each training set
training_set_train_test_split_dict = defaultdict(list)
for training_set_name, training_set in training_set_dict.items():
    X, y = pu_preprocess(training_set, match_col='ref_match')
    X_train, X_val, y_train, y_val = skm.train_test_split(X, y, train_size=0.8, random_state=0)
    training_set_train_test_split_dict[training_set_name] = {'X': X, 'X_train': X_train, 'X_val': X_val, 'y': y, 'y_train': y_train, 'y_val': y_val}

training_set_train_test_split_dict


In [30]:
unbiased_models_dict = defaultdict(list)
unbiased_predictions_dict = defaultdict(list)



for training_set_name, training_set_train_test_split in training_set_train_test_split_dict.items():


In [ ]:
pu_dict[training_set_name][0]

In [ ]:
for set_name in training_sets.keys():
    print(f'Saving {set_name} training set')
    training_set_dict[set_name].to_csv(pu_savedir / f'{set_name}.training_set.csv', index=False)
    print(f'Saving {set_name} unbiased RF models')
    with open(pu_savedir / f'{set_name}.unbiased_models.pkl', 'wb') as f:
        pickle.dump(unbiased_models_dict[set_name], f)
    print(f'Saving {set_name} unbiased predictions')
    pd.DataFrame(unbiased_predictions_dict[set_name][:,1], columns=['unbiased_predictions'], index=training_set_dict[set_name].index).to_csv(pu_savedir / f'{set_name}.unbiased_predictions.csv', index=True)

In [ ]:
for set_name in training_sets.keys():
    print(f'{set_name} unbiased predictions: {sum(unbiased_predictions_dict[set_name] > 0.3)}')

In [ ]:
unbiased_predictions_dict['gridss']

In [ ]:
pd.concat([pd.Series(unbiased_predictions_dict['gridss'][:,1], name='unbiased_predictions_gridss'), joint_bp_df['gridss_QUAL_bnd1']], axis=1).fillna(0).plot.scatter(x='unbiased_predictions_gridss', y='gridss_QUAL_bnd1', alpha=0.1)

In [ ]:
joint_bp_df.plot.box(by='ref_match', column='gridss_QUAL', showfliers=False)